This notebook is to prob (currently with the setting: CGNN-3D, rmsd_cutoff_2, random-k-fold) (linear_probes) for the affinity value and the docking score, and set one as the skyline that would specify the limitation that the embeddings and the linear prob are forcing. <br>

The pipeline to load the $X$ is the same. To make the $y$, one should read the idents of X, load the raw data and sort docking scores/affinities according to the idents from $X$'s idents.

In [2]:
from pathlib import Path
import os
import pandas as pd

from kinodata.data import KinodataDockedAgnostic, KinodataDocked

from prob.paths_and_io import get_project_root, get_exp_dirs, load_X_from_pt, load_out_tensor, save_out_tensor
from prob.prob_config import get_ds_load_config
from prob.prob_models import LINEAR_PROBES
from prob.prob_run import run_cv_search



In [3]:
prob_config = get_ds_load_config()

output_dir = prob_config['output_dir']
target_dir = prob_config['target_dir']
print(f"output_dir: {output_dir}")
print(f"target_dir: {target_dir}")

output_dir: /home/famo00001/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/rmsd_cutoff_2/random-k-fold
target_dir: /home/famo00001/kinodata-3D-affinity-prediction/data/probing/targets


In [4]:
IDS_FILE = "ids.pt"
LAYER_NUM = 3

# Runtime knobs
RANDOM_STATE = 96
N_SPLITS_CV = 5
TEST_SIZE = 0.1


In [5]:
ids = load_out_tensor(output_dir, IDS_FILE)
print(ids.shape, ids.dtype)

torch.Size([41238]) torch.int64


Trying with Docked dataset:

In [ ]:
org_ds = KinodataDocked()
org_ds[0]

HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA',
  scaffold='C1CCC(CC2CCCC(C3CC(C4CCCC4)C4CCCCC34)C2)CC1',
  activity_type='pIC50',
  ident=[1],
  smiles='Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1',
  ligand={
    z=[28],
    x=[28, 12],
    pos=[28, 3],
  },
  pocket={
    z=[652],
    x=[652, 12],
    pos=[652, 3],
  },
  pocket_residue={ x=[85, 23] },
  (ligand, bond, ligand)={
    edge_index=[2, 64],
    edge_attr=[64, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 1308],
    edge_attr=[1308, 4],
  }
)

In [10]:
print(f"{len(org_ds)} , affinity at ident {org_ds[1].ident.item()} : {org_ds[1].y.item()}")
i = org_ds[1].ident.item()

119522 , affinity at ident 17 : 6.0


In [ ]:
org_ds = KinodataDockedAgnostic()
org_ds_df = org_ds.data_frame
org_ds_df.head()

Loading raw data from /home/famo00001/kinodata-3D-affinity-prediction/data/raw...
Reading data frame from /home/famo00001/kinodata-3D-affinity-prediction/data/raw/kinodata_docked_v2.sdf.gz...
Deduping data frame (current size: 140977)...
138286 complexes remain after deduplication.
Checking for missing pocket mol2 files...


100%|██████████| 3551/3551 [00:00<00:00, 9681.12it/s]


Adding pocket sequences...
(138286, 25)


100%|██████████| 138286/138286 [00:00<00:00, 1998929.99it/s]


Exiting with 3552 cached sequences.
(138286, 26)
Converting to data list...
Done!


,docking.posit_probability,docking.chemgauss_score,activities.activity_id,assays.chembl_id,target_dictionary.chembl_id,molecule_dictionary.chembl_id,molecule_dictionary.max_phase,activities.standard_type,activities.standard_units,compound_structures.canonical_smiles,...,UniprotID,similar.klifs_structure_id,similar.fp_similarity,ID,activities.standard_value,docking.predicted_rmsd,molecule,pocket_mol2_file,ident,structure.pocket_sequence
0,0.18,-13.526784,32335,CHEMBL817617,CHEMBL279,CHEMBL69638,nan,pIC50,nM,Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1,...,P35968,5326,0.159664,LIG,5.148742,4.720892,<rdkit.Chem.rdchem.Mol object at 0x748afce30890>,/home/famo00001/kinodata-3D-affinity-predictio...,0,KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVE...
1,0.24,-10.307055,32336,CHEMBL847682,CHEMBL4128,CHEMBL69638,nan,pIC50,nM,Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1,...,Q02763,5553,0.2,32336,5.468521,5.696663,<rdkit.Chem.rdchem.Mol object at 0x748afce30970>,/home/famo00001/kinodata-3D-affinity-predictio...,1,DVIGEG__GQVLKAAIKRM____ELEVLCKLGPNIINLLGAYLAIE...
2,0.18,-11.764866,32680,CHEMBL677833,CHEMBL203,CHEMBL137635,nan,pIC50,nM,CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12,...,P00533,12838,0.212329,LIG,5.031517,4.851336,<rdkit.Chem.rdchem.Mol object at 0x748afce309e0>,/home/famo00001/kinodata-3D-affinity-predictio...,2,KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLIMQ...
3,0.24,-10.21195,32770,CHEMBL674643,CHEMBL203,CHEMBL306988,nan,pIC50,nM,CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)[O-])cc1,...,P00533,786,0.148936,LIG,3.301030,6.134686,<rdkit.Chem.rdchem.Mol object at 0x748afce30a50>,/home/famo00001/kinodata-3D-affinity-predictio...,3,KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLITQ...
4,0.18,-3.132142,32773,CHEMBL675636,CHEMBL203,CHEMBL66879,nan,pKi,nM,O=C([O-])/C=C/c1ccc(O)cc1,...,P00533,15067,0.132075,LIG,3.000000,6.192890,<rdkit.Chem.rdchem.Mol object at 0x748afce30b30>,/home/famo00001/kinodata-3D-affinity-predictio...,4,KVLGS___GTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLIMQ...


In [7]:
skylines = org_ds_df[['ident', 'docking.chemgauss_score', 'activities.standard_value']].rename(columns={'docking.chemgauss_score': 'docking_score', 'activities.standard_value': 'affinity'})
skylines = skylines.set_index('ident')
print(len(skylines))
skylines.head()

138286


,docking_score,affinity
ident,,
0,-13.526784,5.148742
1,-10.307055,5.468521
2,-11.764866,5.031517
3,-10.21195,3.301030
4,-3.132142,3.000000


Loading another target to see how the ident and IDs are handled

In [8]:
test_target_name = "nitrogen_counts"
test_target = load_out_tensor(target_dir, f"{test_target_name}.pt")
print(type(test_target), len(test_target))
print(test_target[0])

<class 'dict'> 119522
4


In [ ]:
# Save skyline targets alongside other probing targets so they can be
# loaded uniformly via load_y_by_ids(target_dir=target_dir, targets_file=...).
for col in skylines.columns:
    save_out_tensor(skylines[col].to_dict(), output_dir=target_dir, filename=f"{col}.pt")
    print(f"Saved {col} -> {target_dir}/{col}.pt")

Test if it can be loaded:

In [ ]:
for col in skylines.columns:
    loaded_col = load_out_tensor(target_dir, f"{col}.pt")
    print(f"Loaded {col} from {target_dir}/{col}.pt")

Test if it can be loaded with ids:

In [12]:
from prob.paths_and_io import load_y_by_ids

loaded_y = load_y_by_ids(output_dir, target_dir=target_dir, targets_file="affinity.pt")
print(type(loaded_y))
loaded_y[i]

<class 'numpy.ndarray'>


9.455932

In [ ]:
# i = 0
expected = skylines.loc[int(ids[i].item()), "affinity"]
assert loaded_y[i] == expected, f"expected {expected} got {loaded_y[i]}"

AssertionError: expected 5.148742 got 9.30103